# Atelier 2 — Profiler les données e-commerce avec Pandas
## Projet fil rouge : BigData Commerce Analytics

**Niveau :** B2 · **Modalité :** individuel ou binôme · 

### Mission
Vous êtes Data Analysts. Vous recevez une table de lignes de panier issue de DummyJSON. Vous devez en comprendre la granularité, vérifier sa structure, mesurer sa qualité et proposer un plan de nettoyage avant stockage dans MongoDB et traitement avec Spark.

### Objectifs pédagogiques
- Charger un CSV et contrôler sa structure.
- Distinguer ligne, panier, utilisateur et produit.
- Mesurer valeurs manquantes, doublons et incohérences.
- Détecter valeurs atypiques et erreurs de typage.
- Documenter les transformations sans détruire les données brutes.

> **Avertissement métier :** le fichier représente des lignes de paniers de démonstration. Ce n'est pas un historique réel de commandes. La colonne `order_date` est synthétique si le fichier a été généré par le script de collecte. Ne l'utilisez pas pour conclure sur une saisonnalité réelle.


## 0. Préparation et chargement
Le fichier complet est créé par `scripts/01_collecte_dummyjson.py` dans `data/processed/commandes_ecommerce.csv`. Si la collecte n'a pas été exécutée, ce notebook utilise le petit échantillon hors ligne de `data/raw/commandes_ecommerce.csv`.


In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
processed = ROOT / "data" / "processed" / "commandes_ecommerce.csv"
sample = ROOT / "data" / "raw" / "commandes_ecommerce.csv"
DATA_FILE = processed if processed.exists() else sample
print("Fichier chargé :", DATA_FILE)
df_brut = pd.read_csv(DATA_FILE)
df = df_brut.copy()
display(df.head())


## 1. Comprendre la granularité

La granularité décrit ce que représente **une ligne**. Ici, une ligne correspond à un produit dans un panier (ligne de panier), et non nécessairement à une commande métier finalisée.

Questions :
1. Une même valeur `order_id` peut-elle apparaître plusieurs fois ? Pourquoi ?
2. Quelle différence entre `cart_id` et `product_id` ?
3. Pourquoi ne faut-il pas additionner `cart_total` sur chaque ligne sans précaution ?


In [ ]:
print("Dimensions (lignes, colonnes) :", df.shape)
print("Nombre de paniers :", df["cart_id"].nunique() if "cart_id" in df else "colonne absente")
print("Nombre de produits :", df["product_id"].nunique() if "product_id" in df else "colonne absente")
print("Nombre d'utilisateurs :", df["user_id"].nunique() if "user_id" in df else "colonne absente")
print("Nombre de lignes :", len(df))


## 2. Types et structure des colonnes

Un identifiant est souvent un entier ou une chaîne, un prix un nombre décimal et une date un objet datetime. Vérifiez les types avant de calculer.


In [ ]:
df.info()
display(df.dtypes.astype(str).to_frame("type détecté"))


In [ ]:
# Conversion explicite des champs numériques et de la date
numeric_cols = ["quantity", "unit_price", "line_total", "discount_percentage",
                "discounted_line_total", "cart_total", "cart_discounted_total"]
for col in numeric_cols:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors="coerce")

if "order_date" in df.columns:
    df["order_date"] = pd.to_datetime(df["order_date"], errors="coerce")

display(df.dtypes.astype(str).to_frame("type après conversion"))


## 3. Valeurs manquantes et doublons

Une valeur manquante n'est pas toujours une erreur : il faut comprendre le sens du champ. Un doublon exact n'est pas la même chose que deux lignes portant le même `order_id` (ce qui est normal si un panier contient plusieurs produits).


In [ ]:
missing = df.isna().sum().sort_values(ascending=False)
missing_pct = (df.isna().mean() * 100).round(2)
qualite_manquants = pd.DataFrame({"valeurs_manquantes":missing, "pourcentage":missing_pct})
display(qualite_manquants)
print("Doublons exacts :", df.duplicated().sum())


In [ ]:
# Doublons sur une clé métier candidate : panier + produit
key_cols = [c for c in ["cart_id", "product_id"] if c in df.columns]
if len(key_cols) == 2:
    print("Doublons sur cart_id + product_id :", df.duplicated(subset=key_cols).sum())


## 4. Statistiques descriptives et valeurs atypiques

Examinez les distributions. Les valeurs extrêmes ne sont pas automatiquement des erreurs : un article cher peut être légitime. Il faut vérifier le contexte et la source.


In [ ]:
display(df.describe(include="all").T)


In [ ]:
for col in [c for c in ["quantity","unit_price","line_total","discount_percentage","discounted_line_total"] if c in df.columns]:
    print(f"\n{col}")
    print(df[col].describe())


## 5. Contrôles de cohérence métier

Vérifiez les règles suivantes :
- quantité strictement positive ;
- prix unitaire positif ou nul selon la règle métier à définir ;
- pourcentage de remise compris entre 0 et 100 ;
- total de ligne proche de quantité × prix unitaire ;
- total remisé inférieur ou égal au total initial ;
- identifiants de produits et paniers renseignés.

Un écart d'arrondi de quelques centimes peut être normal.


In [ ]:
checks = {}
if "quantity" in df:
    checks["quantite_non_positive"] = int((df["quantity"] <= 0).sum())
if "unit_price" in df:
    checks["prix_negatif"] = int((df["unit_price"] < 0).sum())
if "discount_percentage" in df:
    checks["remise_hors_0_100"] = int(((df["discount_percentage"] < 0) | (df["discount_percentage"] > 100)).sum())
if {"quantity","unit_price","line_total"}.issubset(df.columns):
    expected = df["quantity"] * df["unit_price"]
    checks["ecart_total_ligne_superieur_1_centime"] = int(((df["line_total"] - expected).abs() > 0.01).sum())
if {"line_total","discounted_line_total"}.issubset(df.columns):
    checks["total_remise_superieur_total_initial"] = int((df["discounted_line_total"] > df["line_total"] + 0.01).sum())
display(pd.DataFrame(list(checks.items()), columns=["controle","nombre_anomalies"]))


## 6. Analyse exploratoire simple

Les graphiques ci-dessous décrivent les données présentes dans l'instantané. Ils ne doivent pas être interprétés comme des tendances réelles du marché.


In [ ]:
if "category" in df and "line_total" in df:
    ca_categorie = df.groupby("category", dropna=False)["line_total"].sum().sort_values(ascending=False)
    display(ca_categorie.to_frame("total_lignes_panier"))
    ca_categorie.plot(kind="bar", title="Montant brut des lignes de panier par catégorie", ylabel="Montant (unité monétaire API)", xlabel="Catégorie", rot=45)
    plt.tight_layout()
    plt.show()


In [ ]:
if "unit_price" in df:
    df["unit_price"].dropna().plot(kind="hist", bins=20, title="Distribution des prix unitaires", xlabel="Prix unitaire")
    plt.tight_layout()
    plt.show()


## 7. Construire un jeu nettoyé — sans écraser la source

Conservez `df_brut`. Créez une copie et appliquez des règles explicites. Ne remplacez pas toutes les valeurs manquantes par la chaîne `inconnu` : cela peut transformer des colonnes numériques en texte.


In [ ]:
df_clean = df.copy()

# Règles de nettoyage pédagogiques : à adapter après observation des anomalies.
if "quantity" in df_clean:
    df_clean = df_clean[df_clean["quantity"].notna() & (df_clean["quantity"] > 0)]
if "unit_price" in df_clean:
    df_clean = df_clean[df_clean["unit_price"].notna() & (df_clean["unit_price"] >= 0)]
if "cart_id" in df_clean and "product_id" in df_clean:
    df_clean = df_clean.drop_duplicates(subset=["cart_id", "product_id"], keep="first")

print("Avant :", df.shape)
print("Après :", df_clean.shape)


In [ ]:
# Sauvegarder dans data/processed/ (ne pas écraser le CSV brut)
OUT = ROOT / "data" / "processed"
OUT.mkdir(parents=True, exist_ok=True)
clean_path = OUT / "commandes_ecommerce_nettoyees.csv"
df_clean.to_csv(clean_path, index=False)
print("Fichier nettoyé enregistré :", clean_path)


## 8. Conclusion et livrable

Rédigez une courte note de qualité contenant :
1. le nombre de lignes et de colonnes ;
2. la granularité du fichier ;
3. les types à corriger ;
4. le nombre de valeurs manquantes et de doublons ;
5. trois anomalies ou limites observées ;
6. les règles de nettoyage retenues ;
7. les précautions à prendre avant chargement dans MongoDB ou Spark.

### Questions de réflexion
- Pourquoi faut-il garder une copie brute ?
- Pourquoi `cart_total` ne doit-il pas être additionné ligne par ligne ?
- Quelle différence entre qualité technique et pertinence métier ?
- Quelles limites viennent de la source DummyJSON plutôt que de votre code ?

**Livrable :** le notebook exécuté, `commandes_ecommerce_nettoyees.csv` et une note de qualité d'une page.
